In [ ]:
# ============================================================
# COLAB CELL 1 — Install packages
# ============================================================

!pip install ultralytics -q
!pip install albumentations -q
!pip install gdown -q
!pip install pyyaml -q
!pip install timm -q




import torch
print(f" PyTorch : {torch.__version__}")
print(f"   CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"   VRAM   : {vram:.1f} GB")

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

## Summary of Filtered Data

In [ ]:
from pathlib import Path

AUG_DIR = Path("/content/drive/MyDrive/VisDrone_Pipeline/augmented")
DEG_DIR = Path("/content/drive/MyDrive/VisDrone_Pipeline/degraded")
WEIGHTS_DIR = Path("/content/drive/MyDrive/VisDrone_Pipeline/weights")



In [ ]:
FILTERED = Path("/content/drive/MyDrive/VisDrone_Pipeline/filtered")

In [ ]:
# ── Quick verify before moving to SwinIR ─────────────────────
from pathlib import Path

DEG_DIR = Path("/content/drive/MyDrive/VisDrone_Pipeline/degraded")

print("📂 Degraded dataset verification:")
print("="*50)

for split in ["train", "val"]:
    img_dir = DEG_DIR / split / "images"
    lbl_dir = DEG_DIR / split / "labels"

    n_img = len(list(img_dir.glob("*.jpg"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0

    # Check a sample image is not corrupted
    sample_ok = False
    import cv2
    imgs = list(img_dir.glob("*.jpg"))
    if imgs:
        test = cv2.imread(str(imgs[0]))
        sample_ok = test is not None
        h, w = test.shape[:2] if test is not None else (0,0)

    status = "✅" if n_img > 0 and sample_ok else "❌"
    print(f"{status} {split:<6}  images:{n_img:6d}  "
          f"labels:{n_lbl:6d}  "
          f"sample size: {w}×{h}")

print("="*50)
print("\n✅ Ready for Cell 8 — Download SwinIR weights")

In [ ]:
from pathlib import Path

SR2_DIR = Path("/content/drive/MyDrive/VisDrone_Pipeline/sr_2x")
SR4_DIR = Path("/content/drive/MyDrive/VisDrone_Pipeline/sr_4x")
SR8_DIR = Path("/content/drive/MyDrive/VisDrone_Pipeline/sr_8x")

In [ ]:
# ============================================================
# CELL 1 — Check exact structure of swin_sr_2x
# ============================================================

import cv2
from pathlib import Path

SR2_ROOT = Path("/content/drive/MyDrive/VisDrone_Pipeline/sr_2x")

print("="*60)
print(f"  Scanning: {SR2_ROOT}")
print("="*60)

# Show full structure
for item in sorted(SR2_ROOT.rglob("*")):
    if item.is_dir():
        n_jpg = len(list(item.glob("*.jpg")))
        n_png = len(list(item.glob("*.png")))
        n_txt = len(list(item.glob("*.txt")))
        if n_jpg + n_png + n_txt > 0:
            print(f"\n  📁 {item.relative_to(SR2_ROOT)}")
            if n_jpg: print(f"     .jpg : {n_jpg}")
            if n_png: print(f"     .png : {n_png}")
            if n_txt: print(f"     .txt : {n_txt}")

            # Show sample image size
            imgs = list(item.glob("*.jpg")) + list(item.glob("*.png"))
            if imgs:
                s = cv2.imread(str(imgs[0]))
                if s is not None:
                    print(f"     size : {s.shape[1]}×{s.shape[0]}")
                    print(f"     sample: {imgs[0].name}")

            # Show sample label
            txts = list(item.glob("*.txt"))
            if txts:
                content = txts[0].read_text().strip()
                if content:
                    first = content.split("\n")[0]
                    print(f"     label : '{first[:50]}'")

print("\n" + "="*60)

In [ ]:
# ============================================================
# CELL 2 — Set all paths based on structure
#
# If Cell 1 shows:
#   train/images + train/labels  → use Case A (default)
#   images + labels flat         → use Case B
#   jpg/txt in root              → use Case C
# ============================================================

import os, cv2, shutil, yaml, random
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import torch

SR2_ROOT  = Path("/content/drive/MyDrive/VisDrone_Pipeline/sr_2x")
WORK_DIR  = Path("/content/work_sr2x")    # local fast disk
WORK_DIR.mkdir(parents=True, exist_ok=True)

YOLO_NAMES = ["bicycle","car","van","truck",
              "tricycle","awning-tricycle","bus","motor"]

DEVICE     = 0 if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8 if torch.cuda.is_available() and \
             torch.cuda.get_device_properties(0).total_memory/1e9 >= 14 \
             else 4

# ── CASE A — has train/ and val/ subfolders (most likely) ────
TRAIN_IMG = SR2_ROOT / "train" / "images"
TRAIN_LBL = SR2_ROOT / "train" / "labels"
VAL_IMG   = SR2_ROOT / "val"   / "images"
VAL_LBL   = SR2_ROOT / "val"   / "labels"

# ── CASE B — flat images/ and labels/ ─────────────────────
# TRAIN_IMG = SR2_ROOT / "images"
# TRAIN_LBL = SR2_ROOT / "labels"
# VAL_IMG   = SR2_ROOT / "images"
# VAL_LBL   = SR2_ROOT / "labels"

# ── Verify ────────────────────────────────────────────────────
print("="*55)
print("  Path Verification")
print("="*55)
for name, d in [("Train images", TRAIN_IMG),
                ("Train labels", TRAIN_LBL),
                ("Val images",   VAL_IMG),
                ("Val labels",   VAL_LBL)]:
    n      = len(list(d.glob("*"))) if d.exists() else 0
    status = "✅" if n > 0 else "❌"
    print(f"  {status}  {name:<15}: {n} files  [{d}]")

print(f"\n  GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  Batch  : {BATCH_SIZE}")
print(f"  Work   : {WORK_DIR}")

In [ ]:
# ============================================================
# CELL 3 — SAHI-style slicing into 320×320 tiles
#
# Reads from Drive (SR2_ROOT) → writes to local WORK_DIR
# Local disk is 10-20× faster than Drive for I/O
# ============================================================

import cv2, shutil
from pathlib import Path
from tqdm.notebook import tqdm

SLICE_DIR = WORK_DIR / "sliced"
SLICE_H   = 320
SLICE_W   = 320
OVERLAP   = 0.2
MIN_AREA  = 0.3     # box must have ≥30% area inside tile

for split, img_dir, lbl_dir in [
    ("train", TRAIN_IMG, TRAIN_LBL),
    ("val",   VAL_IMG,   VAL_LBL),
]:
    dst_img = SLICE_DIR / split / "images"
    dst_lbl = SLICE_DIR / split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    img_list    = sorted(img_dir.glob("*.jpg")) + \
                  sorted(img_dir.glob("*.png"))
    step_h      = int(SLICE_H * (1 - OVERLAP))
    step_w      = int(SLICE_W * (1 - OVERLAP))
    total_tiles = 0
    skipped     = 0

    print(f"\n🔪 Slicing [{split}]: {len(img_list)} images")

    for img_path in tqdm(img_list, desc=f"Slice {split}"):

        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue

        H, W = img.shape[:2]

        # Load labels for this image
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        boxes    = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().split("\n"):
                p = line.strip().split()
                if len(p) == 5:
                    boxes.append([int(p[0])] +
                                 [float(x) for x in p[1:]])

        # Generate tiles
        for y in range(0, max(1, H - SLICE_H + step_h), step_h):
            for x in range(0, max(1, W - SLICE_W + step_w), step_w):

                y2 = min(y + SLICE_H, H)
                x2 = min(x + SLICE_W, W)
                y1 = max(0, y2 - SLICE_H)
                x1 = max(0, x2 - SLICE_W)

                tile   = img[y1:y2, x1:x2]
                th, tw = tile.shape[:2]

                # Find boxes that overlap this tile
                tile_boxes = []
                for box in boxes:
                    cls, cx, cy, bw, bh = box

                    # Convert normalised → pixel
                    bx1 = (cx - bw/2) * W;  by1 = (cy - bh/2) * H
                    bx2 = (cx + bw/2) * W;  by2 = (cy + bh/2) * H

                    # Intersection with tile
                    ix1 = max(bx1, x1);  iy1 = max(by1, y1)
                    ix2 = min(bx2, x2);  iy2 = min(by2, y2)

                    if ix2 <= ix1 or iy2 <= iy1:
                        continue

                    # Check minimum area inside tile
                    orig_area  = max(1, (bx2-bx1) * (by2-by1))
                    inter_area = (ix2-ix1) * (iy2-iy1)
                    if inter_area / orig_area < MIN_AREA:
                        continue

                    # Convert to tile-local normalised coords
                    ncx = ((ix1 + ix2) / 2 - x1) / tw
                    ncy = ((iy1 + iy2) / 2 - y1) / th
                    nw  = (ix2 - ix1) / tw
                    nh  = (iy2 - iy1) / th

                    # Clamp
                    ncx = max(0.001, min(0.999, ncx))
                    ncy = max(0.001, min(0.999, ncy))
                    nw  = max(0.001, min(0.999, nw))
                    nh  = max(0.001, min(0.999, nh))

                    tile_boxes.append(
                        f"{int(cls)} {ncx:.6f} {ncy:.6f}"
                        f" {nw:.6f} {nh:.6f}")

                # Save tile
                stem     = f"{img_path.stem}_{y1}_{x1}"
                tile_img = dst_img / f"{stem}.jpg"
                tile_lbl = dst_lbl / f"{stem}.txt"
                cv2.imwrite(str(tile_img), tile)
                tile_lbl.write_text("\n".join(tile_boxes))
                total_tiles += 1

    n_img = len(list(dst_img.glob("*.jpg")))
    n_lbl = len(list(dst_lbl.glob("*.txt")))
    n_with_obj = sum(
        1 for f in dst_lbl.glob("*.txt")
        if f.read_text().strip()
    )
    print(f"   ✅ {len(img_list)} images → {total_tiles} tiles")
    print(f"   Images  : {n_img}")
    print(f"   Labels  : {n_lbl}")
    print(f"   Tiles with objects: {n_with_obj} "
          f"({100*n_with_obj/max(1,n_lbl):.1f}%)")
    print(f"   Skipped : {skipped}")

print(f"\n✅ Slicing complete → {SLICE_DIR}")

In [ ]:
# ============================================================
# CELL — Analyse tile contents before fixing
# ============================================================

from pathlib import Path
from tqdm.notebook import tqdm

SLICE_DIR = WORK_DIR / "sliced"

for split in ["train", "val"]:
    lbl_dir = SLICE_DIR / split / "labels"
    img_dir = SLICE_DIR / split / "images"

    all_lbls     = list(lbl_dir.glob("*.txt"))
    total        = len(all_lbls)
    with_obj     = 0
    empty        = 0
    obj_counts   = []

    for lbl in all_lbls:
        content = lbl.read_text().strip()
        lines   = [l for l in content.split("\n") if l.strip()]
        if lines:
            with_obj += 1
            obj_counts.append(len(lines))
        else:
            empty += 1

    import numpy as np
    print(f"\n{'='*55}")
    print(f"  [{split}] Tile Analysis")
    print(f"{'='*55}")
    print(f"  Total tiles        : {total:>8,}")
    print(f"  Tiles WITH objects : {with_obj:>8,}  ({100*with_obj/total:.1f}%)")
    print(f"  Tiles EMPTY        : {empty:>8,}  ({100*empty/total:.1f}%)")
    if obj_counts:
        print(f"  Avg objects/tile   : {np.mean(obj_counts):>8.2f}")
        print(f"  Max objects/tile   : {max(obj_counts):>8}")
    print(f"\n  ⚠️  Imbalance ratio : {empty//max(1,with_obj)}:1 "
          f"(background:object)")
    if empty > with_obj * 3:
        print(f"  ❌ Too many background tiles — training will fail")
    print(f"{'='*55}")

In [ ]:
# ============================================================
# CELL — Re-slice keeping ONLY tiles that contain objects
#
# This is the most effective fix.
# Empty background tiles are discarded entirely.
# Result: every training tile has at least 1 vehicle.
#
# If you want SOME background tiles (helps precision),
# set BACKGROUND_RATIO = 0.5 to keep 1 background per 2 object tiles
# Set BACKGROUND_RATIO = 0 to keep ONLY object tiles
# ============================================================

import cv2, shutil, random
from pathlib import Path
from tqdm.notebook import tqdm

BACKGROUND_RATIO = 0.0   # 0.0 = no background tiles
                          # 0.3 = 1 background per 3 object tiles
                          # 1.0 = equal background and object tiles

SLICE_CLEAN_DIR = WORK_DIR / "sliced_clean"
SLICE_H  = 320
SLICE_W  = 320
OVERLAP  = 0.2
MIN_AREA = 0.3

random.seed(42)

for split, img_dir, lbl_dir in [
    ("train", TRAIN_IMG, TRAIN_LBL),
    ("val",   VAL_IMG,   VAL_LBL),
]:
    dst_img = SLICE_CLEAN_DIR / split / "images"
    dst_lbl = SLICE_CLEAN_DIR / split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    img_list    = sorted(img_dir.glob("*.jpg")) + \
                  sorted(img_dir.glob("*.png"))
    step_h      = int(SLICE_H * (1 - OVERLAP))
    step_w      = int(SLICE_W * (1 - OVERLAP))

    tiles_with_obj = 0
    tiles_bg       = 0
    tiles_bg_kept  = 0
    skipped        = 0

    print(f"\n🔪 Re-slicing [{split}]: {len(img_list)} images")
    print(f"   Background ratio : {BACKGROUND_RATIO}")

    for img_path in tqdm(img_list, desc=f"Clean slice {split}"):
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        H, W = img.shape[:2]

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        boxes    = []
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().split("\n"):
                p = line.strip().split()
                if len(p) == 5:
                    boxes.append([int(p[0])] +
                                 [float(x) for x in p[1:]])

        for y in range(0, max(1, H - SLICE_H + step_h), step_h):
            for x in range(0, max(1, W - SLICE_W + step_w), step_w):
                y2 = min(y + SLICE_H, H)
                x2 = min(x + SLICE_W, W)
                y1 = max(0, y2 - SLICE_H)
                x1 = max(0, x2 - SLICE_W)

                tile   = img[y1:y2, x1:x2]
                th, tw = tile.shape[:2]

                tile_boxes = []
                for box in boxes:
                    cls, cx, cy, bw, bh = box
                    bx1=(cx-bw/2)*W; by1=(cy-bh/2)*H
                    bx2=(cx+bw/2)*W; by2=(cy+bh/2)*H
                    ix1=max(bx1,x1); iy1=max(by1,y1)
                    ix2=min(bx2,x2); iy2=min(by2,y2)
                    if ix2<=ix1 or iy2<=iy1: continue
                    orig  = max(1,(bx2-bx1)*(by2-by1))
                    inter = (ix2-ix1)*(iy2-iy1)
                    if inter/orig < MIN_AREA: continue
                    ncx=((ix1+ix2)/2-x1)/tw
                    ncy=((iy1+iy2)/2-y1)/th
                    nw=(ix2-ix1)/tw; nh=(iy2-iy1)/th
                    ncx=max(0.001,min(0.999,ncx))
                    ncy=max(0.001,min(0.999,ncy))
                    nw=max(0.001,min(0.999,nw))
                    nh=max(0.001,min(0.999,nh))
                    tile_boxes.append(
                        f"{int(cls)} {ncx:.6f} {ncy:.6f}"
                        f" {nw:.6f} {nh:.6f}")

                has_obj = len(tile_boxes) > 0
                stem    = f"{img_path.stem}_{y1}_{x1}"

                if has_obj:
                    # Always keep object tiles
                    cv2.imwrite(str(dst_img/f"{stem}.jpg"), tile)
                    (dst_lbl/f"{stem}.txt").write_text(
                        "\n".join(tile_boxes))
                    tiles_with_obj += 1

                else:
                    # Keep background tile only at BACKGROUND_RATIO
                    tiles_bg += 1
                    if BACKGROUND_RATIO > 0:
                        keep_bg = random.random() < BACKGROUND_RATIO
                        if keep_bg:
                            cv2.imwrite(
                                str(dst_img/f"{stem}.jpg"), tile)
                            (dst_lbl/f"{stem}.txt").write_text("")
                            tiles_bg_kept += 1

    n_img   = len(list(dst_img.glob("*.jpg")))
    n_lbl   = len(list(dst_lbl.glob("*.txt")))
    n_empty = sum(1 for f in dst_lbl.glob("*.txt")
                  if not f.read_text().strip())

    print(f"\n   ✅ Results [{split}]:")
    print(f"   Object tiles kept  : {tiles_with_obj:>8,}")
    print(f"   Background total   : {tiles_bg:>8,}")
    print(f"   Background kept    : {tiles_bg_kept:>8,}")
    print(f"   Total saved        : {n_img:>8,}")
    print(f"   Reduction          : {100*(1-n_img/max(1,tiles_with_obj+tiles_bg)):.1f}% fewer tiles")

print(f"\n✅ Clean slicing done → {SLICE_CLEAN_DIR}")

In [ ]:
# ============================================================
# CELL — Update YAML to use clean sliced dataset
# ============================================================

import yaml
from pathlib import Path

SLICE_CLEAN_DIR = WORK_DIR / "sliced_clean"
YAML_PATH       = WORK_DIR / "visdrone_sr2x_clean.yaml"

n_train = len(list((SLICE_CLEAN_DIR/"train"/"images").glob("*.jpg")))
n_val   = len(list((SLICE_CLEAN_DIR/"val"  /"images").glob("*.jpg")))

# Count object tiles
n_tr_obj = sum(
    1 for f in (SLICE_CLEAN_DIR/"train"/"labels").glob("*.txt")
    if f.read_text().strip()
)
n_va_obj = sum(
    1 for f in (SLICE_CLEAN_DIR/"val"/"labels").glob("*.txt")
    if f.read_text().strip()
)

yaml_data = {
    "path"  : str(SLICE_CLEAN_DIR),
    "train" : "train/images",
    "val"   : "val/images",
    "nc"    : 8,
    "names" : YOLO_NAMES,
}
with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print("✅ Updated YAML:")
print(open(YAML_PATH).read())
print(f"   Train: {n_train:,} tiles  ({n_tr_obj:,} with objects)")
print(f"   Val  : {n_val:,} tiles  ({n_va_obj:,} with objects)")
print(f"\n   Object tile % train: {100*n_tr_obj/max(1,n_train):.1f}%")
print(f"   Object tile % val  : {100*n_va_obj/max(1,n_val):.1f}%")

In [ ]:
# ============================================================
# CELL — Train RT-DETR on clean object-only sliced tiles
# ============================================================

import torch
from ultralytics import RTDETR

_orig = torch.load
def patched_load(f, *args, **kwargs):
    kwargs["weights_only"] = False
    return _orig(f, *args, **kwargs)
torch.load = patched_load

model = RTDETR("rtdetr-l.pt")

print(f"🚀 Training on CLEAN SR 2× tiles")
print(f"   Train: {n_train:,} tiles  Val: {n_val:,} tiles")
print(f"   Batch: {BATCH_SIZE}  Device: {DEVICE}")

model.train(
    data          = str(YAML_PATH),
    epochs        = 50,
    imgsz         = 320,
    batch         = BATCH_SIZE,
    lr0           = 1e-4,
    optimizer     = "AdamW",
    patience      = 15,
    warmup_epochs = 5,
    device        = DEVICE,
    project       = str(WORK_DIR / "runs"),
    name          = "rtdetr_sr2x_clean",
    exist_ok      = True,
    amp           = True,
    save          = True,
    save_period   = 5,
    workers       = 2,
    verbose       = True,
    box           = 7.5,
    cls           = 0.5,
    dfl           = 1.5,
)

BEST_WEIGHTS = WORK_DIR / "runs" / "rtdetr_sr2x_clean" / \
               "weights" / "best.pt"
print(f"\n✅ Training done → {BEST_WEIGHTS}")

In [ ]:
# ============================================================
# CELL 6 — Evaluate mAP on SR 2× val set
# ============================================================

from ultralytics import RTDETR
from pathlib import Path

BEST_WEIGHTS = WORK_DIR / "runs" / "rtdetr_sr2x" / "weights" / "best.pt"

if not BEST_WEIGHTS.exists():
    # Search for it
    import os
    for root, dirs, files in os.walk(WORK_DIR):
        for f in files:
            if f == "best.pt":
                BEST_WEIGHTS = Path(root) / f
                print(f"Found: {BEST_WEIGHTS}")
                break

model_eval = RTDETR(str(BEST_WEIGHTS))

metrics = model_eval.val(
    data    = str(YAML_PATH),
    split   = "val",
    imgsz   = 320,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    verbose = True,
)

MAP50    = metrics.box.map50
MAP5095  = metrics.box.map
PREC     = metrics.box.mp
RECALL   = metrics.box.mr

print("\n" + "="*55)
print("  RT-DETR  SR 2×  RESULTS")
print("="*55)
print(f"  mAP@50    : {MAP50:.4f}")
print(f"  mAP@50-95 : {MAP5095:.4f}")
print(f"  Precision : {PREC:.4f}")
print(f"  Recall    : {RECALL:.4f}")
print("="*55)

# Per-class
print(f"\n  Per-class mAP@50:")
print(f"  {'Class':<20} {'mAP@50':>10}")
print("  " + "-"*32)
try:
    for i, ap in enumerate(metrics.box.ap50):
        print(f"  {YOLO_NAMES[i]:<20} {ap:>10.4f}")
except:
    print("  (not available)")

In [ ]:
new_checkpoint_path = WORK_DIR / "runs" / "rtdetr_sr2x_clean" / "weights" / "my_custom_checkpoint.pt"
model_eval.save(str(new_checkpoint_path))
print(f"✅ Model saved to: {new_checkpoint_path}")

In [ ]:
# ============================================================
# CELL 7 — Training curves from results.csv
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CSV_PATH = WORK_DIR / "runs" / "rtdetr_sr2x" / "results.csv"

if not CSV_PATH.exists():
    print(f"❌ results.csv not found at {CSV_PATH}")
else:
    df      = pd.read_csv(CSV_PATH)
    df.columns = df.columns.str.strip()
    epochs  = df["epoch"]

    print("Columns:", list(df.columns))

    # Find columns
    map50_col  = next((c for c in df.columns
                       if "mAP50" in c and "95" not in c), None)
    map5095_col= next((c for c in df.columns
                       if "mAP50-95" in c), None)
    box_loss_col= next((c for c in df.columns
                        if "box_loss" in c and "val" in c), None)
    cls_loss_col= next((c for c in df.columns
                        if "cls_loss" in c and "val" in c), None)
    tbox_col   = next((c for c in df.columns
                       if "box_loss" in c and "train" in c), None)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor("#F8F9FA")
    axes = axes.flatten()

    def plot_col(ax, col, title, color, ylabel="Score"):
        if col and col in df.columns:
            ax.plot(epochs, df[col], color=color,
                    linewidth=2.5, marker="o", markersize=3)
            ax.set_title(title, fontweight="bold", fontsize=11)
            ax.set_xlabel("Epoch", fontsize=10)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.grid(linestyle="--", alpha=0.5)
            best_val = df[col].max() if "mAP" in col \
                       else df[col].min()
            best_ep  = df[col].idxmax() if "mAP" in col \
                       else df[col].idxmin()
            ax.axvline(epochs[best_ep], color="red",
                       linestyle=":", alpha=0.7,
                       label=f"Best: {best_val:.4f} @ ep{epochs[best_ep]}")
            ax.legend(fontsize=9)
        else:
            ax.text(0.5, 0.5, f"{title}\n(not in CSV)",
                    ha="center", va="center",
                    transform=ax.transAxes, fontsize=10,
                    color="gray")
            ax.axis("off")

    plot_col(axes[0], map50_col,   "mAP@50",         "#2ECC71")
    plot_col(axes[1], map5095_col, "mAP@50-95",      "#27AE60")
    plot_col(axes[2], box_loss_col,"Val Box Loss",    "#E74C3C", "Loss")
    plot_col(axes[3], cls_loss_col,"Val Class Loss",  "#E67E22", "Loss")
    plot_col(axes[4], tbox_col,    "Train Box Loss",  "#3498DB", "Loss")

    # Precision + Recall on axes[5]
    prec_col = next((c for c in df.columns if "precision" in c.lower()), None)
    rec_col  = next((c for c in df.columns if "recall"    in c.lower()), None)
    if prec_col and rec_col:
        axes[5].plot(epochs, df[prec_col], color="#9B59B6",
                     linewidth=2.5, label="Precision")
        axes[5].plot(epochs, df[rec_col],  color="#1ABC9C",
                     linewidth=2.5, label="Recall")
        axes[5].set_title("Precision & Recall",
                          fontweight="bold", fontsize=11)
        axes[5].set_xlabel("Epoch"); axes[5].set_ylabel("Score")
        axes[5].legend(fontsize=9)
        axes[5].grid(linestyle="--", alpha=0.5)
    else:
        axes[5].axis("off")

    plt.suptitle(
        "RT-DETR Training Curves — SR 2× Sliced Dataset\n"
        "VisDrone Vehicles | Rotational Aug | Blur+Noise | SwinIR 2×",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(str(WORK_DIR / "training_curves_sr2x.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Saved → training_curves_sr2x.png")

In [ ]:
# ============================================================
# CELL 8 — Per-class results visualization
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Per-class values from evaluation
try:
    per_class_map50 = list(metrics.box.ap50)
    per_class_prec  = list(metrics.box.p)
    per_class_rec   = list(metrics.box.r)
except:
    # Fallback if attributes differ
    per_class_map50 = [0] * 8
    per_class_prec  = [0] * 8
    per_class_rec   = [0] * 8
    print("⚠️  Per-class metrics not available — using zeros")

COLORS = ["#3498DB","#2ECC71","#E74C3C","#F39C12",
          "#9B59B6","#1ABC9C","#E67E22","#2C3E50"]

x = np.arange(len(YOLO_NAMES))
w = 0.28

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.patch.set_facecolor("#F8F9FA")

# ── mAP@50 per class ──────────────────────────────────────────
bars = axes[0].bar(x, per_class_map50, color=COLORS, edgecolor="k")
for bar, val in zip(bars, per_class_map50):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.005,
                 f"{val:.3f}", ha="center",
                 fontsize=9, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(YOLO_NAMES, rotation=30, fontsize=9)
axes[0].set_ylabel("mAP@50")
axes[0].set_title("Per-Class mAP@50",
                  fontweight="bold", fontsize=12)
axes[0].set_ylim(0, max(per_class_map50)*1.3 + 0.05)
axes[0].axhline(MAP50, color="red", linestyle="--",
                linewidth=1.5,
                label=f"Overall mAP@50 = {MAP50:.4f}")
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# ── Precision per class ───────────────────────────────────────
bars2 = axes[1].bar(x, per_class_prec, color=COLORS, edgecolor="k")
for bar, val in zip(bars2, per_class_prec):
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.005,
                 f"{val:.3f}", ha="center",
                 fontsize=9, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(YOLO_NAMES, rotation=30, fontsize=9)
axes[1].set_ylabel("Precision")
axes[1].set_title("Per-Class Precision",
                  fontweight="bold", fontsize=12)
axes[1].set_ylim(0, 1.15)
axes[1].axhline(PREC, color="red", linestyle="--",
                linewidth=1.5,
                label=f"Overall P = {PREC:.4f}")
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

# ── Recall per class ──────────────────────────────────────────
bars3 = axes[2].bar(x, per_class_rec, color=COLORS, edgecolor="k")
for bar, val in zip(bars3, per_class_rec):
    axes[2].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.005,
                 f"{val:.3f}", ha="center",
                 fontsize=9, fontweight="bold")
axes[2].set_xticks(x)
axes[2].set_xticklabels(YOLO_NAMES, rotation=30, fontsize=9)
axes[2].set_ylabel("Recall")
axes[2].set_title("Per-Class Recall",
                  fontweight="bold", fontsize=12)
axes[2].set_ylim(0, 1.15)
axes[2].axhline(RECALL, color="red", linestyle="--",
                linewidth=1.5,
                label=f"Overall R = {RECALL:.4f}")
axes[2].legend(fontsize=9)
axes[2].grid(axis="y", linestyle="--", alpha=0.4)

plt.suptitle(
    f"RT-DETR Per-Class Results — SR 2× Dataset\n"
    f"Overall: mAP@50={MAP50:.4f}  mAP@50-95={MAP5095:.4f}"
    f"  P={PREC:.4f}  R={RECALL:.4f}",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(str(WORK_DIR/"per_class_results_sr2x.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → per_class_results_sr2x.png")

In [ ]:
# ============================================================
# CELL 9 — Run inference on sample val tiles + visualise
# ============================================================

import cv2, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import RTDETR

model_infer = RTDETR(str(BEST_WEIGHTS))

VAL_IMG_DIR = SLICE_DIR / "val" / "images"
VAL_LBL_DIR = SLICE_DIR / "val" / "labels"

# Pick 6 tiles that have objects
tiles_with_obj = [
    p for p in sorted(VAL_IMG_DIR.glob("*.jpg"))
    if (VAL_LBL_DIR / (p.stem + ".txt")).exists()
    and (VAL_LBL_DIR / (p.stem + ".txt")).read_text().strip()
]
samples = random.sample(tiles_with_obj,
                        min(6, len(tiles_with_obj)))

COLORS_DET = {
    0:"#00FFFF", 1:"#00FF00", 2:"#FF6600",
    3:"#FF0000", 4:"#FF00FF", 5:"#FFFF00",
    6:"#0088FF", 7:"#FF88FF"
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.patch.set_facecolor("#1A1A2E")
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W    = img_bgr.shape[:2]

    # Run inference
    results = model_infer.predict(
        str(img_path), conf=0.25,
        iou=0.45, imgsz=320,
        device=DEVICE, verbose=False
    )
    boxes   = results[0].boxes

    ax.imshow(img_rgb)

    # Draw GT boxes (green dashed)
    lbl_path = VAL_LBL_DIR / (img_path.stem + ".txt")
    for line in lbl_path.read_text().strip().split("\n"):
        p = line.strip().split()
        if len(p) != 5: continue
        cls = int(p[0])
        cx,cy,bw,bh = float(p[1]),float(p[2]),float(p[3]),float(p[4])
        x1 = (cx-bw/2)*W;  y1 = (cy-bh/2)*H
        rect = patches.Rectangle(
            (x1,y1), bw*W, bh*H,
            linewidth=1.5, edgecolor="lime",
            facecolor="none", linestyle="--")
        ax.add_patch(rect)

    # Draw predicted boxes (coloured solid)
    n_det = 0
    for box in boxes:
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        color = COLORS_DET.get(cls, "white")
        rect  = patches.Rectangle(
            (x1,y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1-3,
                f"{YOLO_NAMES[cls]} {conf:.2f}",
                color=color, fontsize=7, fontweight="bold",
                bbox=dict(facecolor="black",
                          alpha=0.5, pad=1))
        n_det += 1

    ax.set_title(
        f"{img_path.name[:30]}\n"
        f"Pred:{n_det} detections",
        color="white", fontsize=8
    )
    for spine in ax.spines.values():
        spine.set_edgecolor("#3498DB")
        spine.set_linewidth(2)
    ax.tick_params(left=False, bottom=False,
                   labelleft=False, labelbottom=False)

plt.suptitle(
    "RT-DETR Inference — SR 2× Val Tiles\n"
    "Green dashed = Ground Truth  |  Coloured solid = Predicted",
    fontsize=13, fontweight="bold", color="white"
)
plt.tight_layout()
plt.savefig(str(WORK_DIR/"inference_samples_sr2x.png"),
            dpi=150, bbox_inches="tight",
            facecolor="#1A1A2E")
plt.show()
print("✅ Saved → inference_samples_sr2x.png")

In [ ]:
# ============================================================
# CELL 10 — Final summary dashboard
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(20, 8))
fig.patch.set_facecolor("#F8F9FA")

gs = gridspec.GridSpec(1, 3, figure=fig,
                       width_ratios=[1, 1.5, 1])

# ── LEFT — Overall metrics ────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
metric_names  = ["mAP@50", "mAP@50-95", "Precision", "Recall"]
metric_values = [MAP50, MAP5095, PREC, RECALL]
metric_colors = ["#2ECC71","#27AE60","#3498DB","#E74C3C"]

bars = ax1.barh(metric_names, metric_values,
                color=metric_colors, edgecolor="k")
for bar, val in zip(bars, metric_values):
    ax1.text(val + 0.01, bar.get_y()+bar.get_height()/2,
             f"{val:.4f}", va="center",
             fontsize=12, fontweight="bold")
ax1.set_xlim(0, 1.1)
ax1.set_title("Overall Metrics\nRT-DETR SR 2×",
              fontweight="bold", fontsize=12)
ax1.grid(axis="x", linestyle="--", alpha=0.4)

# ── MIDDLE — Per-class mAP@50 horizontal bar ──────────────────
ax2 = fig.add_subplot(gs[1])
y   = np.arange(len(YOLO_NAMES))
bars2 = ax2.barh(y, per_class_map50,
                  color=COLORS, edgecolor="k")
for bar, val in zip(bars2, per_class_map50):
    ax2.text(val + 0.005,
             bar.get_y()+bar.get_height()/2,
             f"{val:.3f}", va="center",
             fontsize=10, fontweight="bold")
ax2.set_yticks(y)
ax2.set_yticklabels(YOLO_NAMES, fontsize=10)
ax2.set_xlim(0, max(per_class_map50)*1.4 + 0.05)
ax2.axvline(MAP50, color="red", linestyle="--",
            linewidth=2, label=f"Mean={MAP50:.4f}")
ax2.set_title("Per-Class mAP@50",
              fontweight="bold", fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(axis="x", linestyle="--", alpha=0.4)

# ── RIGHT — Dataset info panel ────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.axis("off")

info_lines = [
    ("Model",    "RT-DETR-L"),
    ("Dataset",  "VisDrone SR 2×"),
    ("SR Method","SwinIR 2×"),
    ("Classes",  "8 vehicle types"),
    ("Train tiles", f"{n_train:,}"),
    ("Val tiles",   f"{n_val:,}"),
    ("Tile size",   "320×320 px"),
    ("Overlap",     "20%"),
    ("Epochs",      "50"),
    ("Batch",       str(BATCH_SIZE)),
    ("Optimizer",   "AdamW"),
    ("LR",          "1e-4"),
    ("mAP@50",      f"{MAP50:.4f}"),
    ("mAP@50-95",   f"{MAP5095:.4f}"),
]

y_pos = 0.97
for label, value in info_lines:
    ax3.text(0.05, y_pos, label + ":",
             transform=ax3.transAxes,
             fontsize=10, fontweight="bold",
             color=BLUE)
    ax3.text(0.55, y_pos, value,
             transform=ax3.transAxes,
             fontsize=10, color="black")
    y_pos -= 0.068

ax3.set_title("Experiment Config",
              fontweight="bold", fontsize=12)

plt.suptitle(
    "RT-DETR Final Dashboard — SwinIR SR 2× Dataset\n"
    "VisDrone Vehicles | Rotational Aug | Blur+Noise | SAHI Sliced Training",
    fontsize=13, fontweight="bold", y=1.02
)

plt.tight_layout()
plt.savefig(str(WORK_DIR/"final_dashboard_sr2x.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved → final_dashboard_sr2x.png")

# ── Final printed summary ─────────────────────────────────────
print("\n" + "="*55)
print("  FINAL RESULTS — RT-DETR on SwinIR SR 2×")
print("="*55)
print(f"  mAP@50    : {MAP50:.4f}")
print(f"  mAP@50-95 : {MAP5095:.4f}")
print(f"  Precision : {PREC:.4f}")
print(f"  Recall    : {RECALL:.4f}")
print("="*55)
print(f"\n  Output files in: {WORK_DIR}")
print(f"  training_curves_sr2x.png")
print(f"  per_class_results_sr2x.png")
print(f"  inference_samples_sr2x.png")
print(f"  final_dashboard_sr2x.png")